# Prototype and Iterate on LLM Applications

Register a prototype project, auto-evaluate spans with EvalTags, iterate with versioned prompts, compare versions, and choose the winner before deploying to production.

By the end of this notebook you will have registered a prototype project with automatic span evaluation, run an OpenAI app against it, iterated with a second prompt version, compared both versions in the Prototype dashboard, and chosen a winner.

**Prerequisites:**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` ([Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+
- OpenAI API key

## Install

In [ ]:
%pip install fi-instrumentation-otel traceAI-openai openai --quiet

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"          # Replace with your key
os.environ["FI_SECRET_KEY"] = "your-secret-key"    # Replace with your key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # Replace with your key

## What is Prototype?

Prototype lets you test different LLM configurations, prompts, and parameters in a controlled environment before deploying to production. Each run is a **version**: you compare versions side by side on evaluation scores, cost, and latency, then choose a winner.

## Step 1: Register a prototype project (Version 1)

`register()` creates a tracer provider connected to FutureAGI. Setting `project_type=ProjectType.EXPERIMENT` creates a Prototype project. The `project_version_name` tags all traces from this run as a distinct version you can compare later.

`EvalTag` objects define which evaluations run automatically on every matching span, with no manual eval calls needed.

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import (
    ProjectType,
    EvalName,
    EvalTag,
    EvalTagType,
    EvalSpanKind,
    ModelChoices,
)

trace_provider = register(
    project_type=ProjectType.EXPERIMENT,
    project_name="support-bot-prototype",
    project_version_name="v1-baseline",
    eval_tags=[
        EvalTag(
            eval_name=EvalName.TONE,
            type=EvalTagType.OBSERVATION_SPAN,
            value=EvalSpanKind.LLM,
            model=ModelChoices.TURING_FLASH,
            custom_eval_name="tone_check",
            mapping={
                "input": "llm.output_messages.0.message.content",
            },
        ),
        EvalTag(
            eval_name=EvalName.COMPLETENESS,
            type=EvalTagType.OBSERVATION_SPAN,
            value=EvalSpanKind.LLM,
            model=ModelChoices.TURING_FLASH,
            custom_eval_name="completeness_check",
            mapping={
                "input": "llm.input_messages.1.message.content",
                "output": "llm.output_messages.0.message.content",
            },
        ),
    ],
)

## Step 2: Instrument and run your app

Patch the OpenAI client with `OpenAIInstrumentor` so every API call is automatically traced and evaluated against your `EvalTag` configuration.

In [ ]:
from traceai_openai import OpenAIInstrumentor
from openai import OpenAI

OpenAIInstrumentor().instrument(tracer_provider=trace_provider)

client = OpenAI()

questions = [
    "How do I reset my password?",
    "What is your refund policy?",
    "Can I upgrade my plan mid-cycle?",
]

for q in questions:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful customer support agent. Answer concisely."},
            {"role": "user", "content": q},
        ],
    )
    print(f"Q: {q}")
    print(f"A: {response.choices[0].message.content}\n")

trace_provider.force_flush()

## Step 3: View results in the Prototype dashboard

Go to [app.futureagi.com](https://app.futureagi.com), select **Prototype** (left sidebar under BUILD), and click your project **support-bot-prototype** to see version **v1-baseline**.

The dashboard shows:
- Every traced span with its input, output, token count, and latency
- Evaluation scores from your `EvalTag` configuration (`tone_check` and `completeness_check`) displayed alongside each span

## Step 4: Create Version 2 with a different prompt

This is where rapid iteration happens. Register a new version with a different `project_version_name` and run the same queries with an improved prompt.

> **Important:** Each call to `register()` creates a new tracer provider. **Restart the kernel** before running the cells below — do not call `register()` twice in the same process.

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"          # Replace with your key
os.environ["FI_SECRET_KEY"] = "your-secret-key"    # Replace with your key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # Replace with your key

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import (
    ProjectType,
    EvalName,
    EvalTag,
    EvalTagType,
    EvalSpanKind,
    ModelChoices,
)
from traceai_openai import OpenAIInstrumentor
from openai import OpenAI

trace_provider_v2 = register(
    project_type=ProjectType.EXPERIMENT,
    project_name="support-bot-prototype",
    project_version_name="v2-detailed",
    eval_tags=[
        EvalTag(
            eval_name=EvalName.TONE,
            type=EvalTagType.OBSERVATION_SPAN,
            value=EvalSpanKind.LLM,
            model=ModelChoices.TURING_FLASH,
            custom_eval_name="tone_check",
            mapping={
                "input": "llm.output_messages.0.message.content",
            },
        ),
        EvalTag(
            eval_name=EvalName.COMPLETENESS,
            type=EvalTagType.OBSERVATION_SPAN,
            value=EvalSpanKind.LLM,
            model=ModelChoices.TURING_FLASH,
            custom_eval_name="completeness_check",
            mapping={
                "input": "llm.input_messages.1.message.content",
                "output": "llm.output_messages.0.message.content",
            },
        ),
    ],
)

OpenAIInstrumentor().instrument(tracer_provider=trace_provider_v2)

client = OpenAI()

questions = [
    "How do I reset my password?",
    "What is your refund policy?",
    "Can I upgrade my plan mid-cycle?",
]

for q in questions:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a knowledgeable customer support agent. "
                    "Provide detailed, step-by-step answers. "
                    "Include any relevant edge cases or exceptions. "
                    "End with a follow-up question to confirm the issue is resolved."
                ),
            },
            {"role": "user", "content": q},
        ],
    )
    print(f"Q: {q}")
    print(f"A: {response.choices[0].message.content}\n")

trace_provider_v2.force_flush()

## Step 5: Compare versions in the dashboard

Back in the Prototype dashboard, your project now shows two versions: **v1-baseline** and **v2-detailed**.

Click any version to see its individual traces and eval scores. The project overview shows aggregate metrics across all versions — average eval scores, latency, token usage, and cost — so you can compare at a glance.

## Step 6: Choose the winner

Once you have compared evaluation scores, latency, and cost across versions, choose a winner.

1. Go to **Prototype** → click your project
2. Click **Choose Winner**
3. Adjust the importance sliders (0–10) for each metric — evaluation scores, average response time, completion tokens, total tokens
4. Click **Choose Winner** to rank all versions

The version with the highest weighted score is selected as the winner.

## What you built

- Registered a Prototype project with `ProjectType.EXPERIMENT` and automatic span evaluation via `EvalTag`
- Ran a baseline OpenAI app (v1) and saw tone and completeness scores in the dashboard
- Iterated with a new prompt version (v2) using a different `project_version_name`
- Compared both versions on eval scores, latency, and cost in the Prototype dashboard
- Chose the winning version using weighted metric comparison

### Next steps

- [Prototype Overview](https://docs.futureagi.com/future-agi/get-started/prototype/overview) — Full Prototype documentation
- [Prototype Evals Reference](https://docs.futureagi.com/future-agi/get-started/prototype/evals) — All available EvalTag configurations
- [Compare Prompts in Experiments](https://docs.futureagi.com/cookbook/quickstart/experimentation-compare-prompts) — Dataset-level prompt comparison
- [Manual Tracing](https://docs.futureagi.com/cookbook/quickstart/manual-tracing) — Add custom spans, user context, and metadata